<a href="https://colab.research.google.com/github/Cicero-Farias/AED/blob/main/AED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt


def le_instancia(caminho):
    """
    Lê uma instância contendo N flocos de neve.
    """

    with open(caminho) as f:

        n = int(f.readline())

        flocos = []

        for _ in range(n):

            floco = list(
                map(
                    int,
                    f.readline().split()
                )
            )

            flocos.append(floco)

    return flocos


def sao_gemeos(a, b):
    """
    Verifica se b é uma rotação de a.
    """

    for deslocamento in range(6):

        rotacao = (
            a[deslocamento:]
            +
            a[:deslocamento]
        )

        if rotacao == b:

            return True

    return False


def existe_par_gemeo_ingenuo(flocos):
    """
    Solução ingênua:
    compara todos os pares de flocos.
    """

    n = len(flocos)

    for i in range(n):

        for j in range(i + 1, n):

            if sao_gemeos(
                flocos[i],
                flocos[j]
            ):

                return i, j

    return None


def chave_canonica(floco):
    """
    Retorna a menor rotação do floco em ordem lexicográfica.
    """

    rotacoes = []

    for deslocamento in range(6):

        rotacao = tuple(
            floco[deslocamento:]
            +
            floco[:deslocamento]
        )

        rotacoes.append(rotacao)

    return min(rotacoes)


def existe_par_gemeo_hash(flocos):
    """
    Solução utilizando tabela hash.
    """

    vistos = {}

    for j, floco in enumerate(flocos):

        chave = chave_canonica(floco)

        if chave in vistos:

            i = vistos[chave]

            return i, j

        vistos[chave] = j

    return None


def benchmark(caminho, algoritmo, repeticoes=3):
    """
    Executa o algoritmo algumas vezes e retorna
    o menor tempo observado.
    """

    flocos = le_instancia(caminho)

    tempos = []

    resultado = None

    for _ in range(repeticoes):

        inicio = time.perf_counter()

        resultado = algoritmo(flocos)

        fim = time.perf_counter()

        tempos.append(
            fim - inicio
        )

    return min(tempos), resultado


def executar_benchmark(instancias):

    resultados = []

    print(
        f"{'Instância':35s} "
        f"{'N':>8s} "
        f"{'Ingênuo (s)':>15s} "
        f"{'Hash (s)':>15s}"
    )

    for nome in instancias:

        flocos = le_instancia(nome)

        n = len(flocos)

        t_ingenuo, r1 = benchmark(
            nome,
            existe_par_gemeo_ingenuo
        )

        t_hash, r2 = benchmark(
            nome,
            existe_par_gemeo_hash
        )

        assert (
            (r1 is None)
            ==
            (r2 is None)
        ), "As soluções discordam!"

        resultados.append({
            "N": n,
            "Instância": nome,
            "Ingênuo (s)": t_ingenuo,
            "Hash (s)": t_hash
        })

        print(
            f"{nome:35s} "
            f"{n:8d} "
            f"{t_ingenuo:15.6f} "
            f"{t_hash:15.6f}"
        )

    return pd.DataFrame(resultados)


if __name__ == "__main__":

    instancias = [
        "floco_semgemeos_500.txt",
        "floco_semgemeos_1000.txt",
        "floco_semgemeos_2000.txt",
        "floco_semgemeos_4000.txt",
        "floco_semgemeos_8000.txt",
        "floco_semgemeos_16000.txt"
    ]

    df_resultados = executar_benchmark(
        instancias
    )

    print("\nTabela de resultados:\n")

    print(df_resultados)

    df_resultados.to_csv(
        "resultados_benchmark.csv",
        index=False
    )

    plt.figure(figsize=(10, 6))

    plt.plot(
        df_resultados["N"],
        df_resultados["Ingênuo (s)"],
        marker="o",
        label="Busca ingênua"
    )

    plt.plot(
        df_resultados["N"],
        df_resultados["Hash (s)"],
        marker="o",
        label="Tabela hash"
    )

    plt.xlabel("Número de flocos (N)")

    plt.ylabel("Tempo de execução (segundos)")

    plt.title(
        "Busca ingênua vs. tabela hash"
    )

    plt.grid(True)

    plt.legend()

    plt.savefig(
        "grafico_tempos.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()